# Tau Event Reaction Extraction


In [1]:
from pathlib import Path
import importlib.util
import json
import math

import pandas as pd
from icecube import dataio, dataclasses, icetray, LeptonInjector

PATHS_PY = Path('/project/def-nahee/kbas/Graphnet-Applications/Metadata/paths.py')


def load_paths():
    spec = importlib.util.spec_from_file_location('paths', PATHS_PY)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod


paths = load_paths()
Muon_RAW_DIR = Path(paths.STRING340MC_I3['full_geometry']['Muon']['path'])
Muon_RAW_DIR


PosixPath('/project/6008051/pone_simulation/MC000002-nu_mu-2_7-LeptonInjector-PROPOSAL-clsim-v10/Photon')

In [2]:
import random

files = list(Muon_RAW_DIR.glob("*.i3*"))
target_file = random.choice(files)
target_file

PosixPath('/project/6008051/pone_simulation/MC000002-nu_mu-2_7-LeptonInjector-PROPOSAL-clsim-v10/Photon/cls_1262.i3')

In [3]:
from icecube import dataio, icetray

i3_file = dataio.I3File(str(target_file), "r")

try:
    selected_frame = None

    while i3_file.more():
        frame = i3_file.pop_frame()

        if frame.Stop == icetray.I3Frame.DAQ:
            selected_frame = frame
            break

    if selected_frame is None:
        raise LookupError(f"No DAQ frame found in {target_file}")

finally:
    i3_file.close()

selected_frame

In [4]:
def reactions_json_from_frame(frame, tree_key='I3MCTree_postprop', skip_self_copies=True, xyz_tol=1e-6):
    tree = frame[tree_key]
    rows = []

    def particle_name(p):
        return str(p.type).split('.')[-1]

    def particle_xyz(p):
        return [float(p.pos.x), float(p.pos.y), float(p.pos.z)]

    def xyz_group_key(p):
        return tuple(round(v / xyz_tol) for v in particle_xyz(p))

    def clean_length(p):
        length = float(p.length)
        return None if math.isnan(length) else length

    def same_xyz(a, b):
        return all(abs(x - y) < xyz_tol for x, y in zip(a, b))

    for parent in tree.pre_order_iter():
        children = list(tree.children(parent))
        if not children:
            continue

        physical_children = []
        for child in children:
            is_self_copy = (
                parent.pdg_encoding == child.pdg_encoding
                and abs(float(parent.energy) - float(child.energy)) < 1e-6
            )
            if skip_self_copies and is_self_copy:
                continue
            physical_children.append(child)

        if not physical_children:
            continue

        parent_name = particle_name(parent)
        is_nu = 'Nu' in parent_name

        child_groups = {}
        for child in physical_children:
            child_groups.setdefault(xyz_group_key(child), []).append(child)

        for group_children in child_groups.values():
            has_interaction_product = any(
                int(child.pdg_encoding) != int(parent.pdg_encoding)
                for child in group_children
            )
            if not is_nu and not has_interaction_product:
                continue

            vertex_xyz = particle_xyz(group_children[0])
            xyz_key = 'interaction_xyz' if is_nu else 'prop_xyz'

            if is_nu:
                share_key = 'everyone_share_interaction_xyz'
                share_particles = [parent] + group_children
            else:
                share_key = 'children_share_prop_xyz'
                share_particles = group_children

            all_same_xyz = all(same_xyz(vertex_xyz, particle_xyz(p)) for p in share_particles)

            row = {
                'parent': parent_name,
                'parent_pdg': int(parent.pdg_encoding),
                'parent_creation_energy': float(parent.energy),

                'children': [particle_name(c) for c in group_children],
                'children_pdg': [int(c.pdg_encoding) for c in group_children],
                'children_creation_energy': [float(c.energy) for c in group_children],
                'children_length': [clean_length(c) for c in group_children],

                xyz_key: vertex_xyz,
                share_key: all_same_xyz,

                'reaction': (
                    f'{parent_name} -> '
                    + ' + '.join(particle_name(c) for c in group_children)
                ),
            }

            if not is_nu:
                row['parent_length'] = clean_length(parent)
                row['parent_creation_point_xyz'] = particle_xyz(parent)

            rows.append(row)

    return json.dumps(rows)


In [7]:
reactions_json = reactions_json_from_frame(selected_frame)
import json
from pprint import pprint

pprint(json.loads(reactions_json), width=120)

[{'children': ['MuMinus', 'Hadrons'],
  'children_creation_energy': [7025.321571476298, 1301.640021744926],
  'children_length': [4532.132120056887, None],
  'children_pdg': [13, -2000001006],
  'everyone_share_interaction_xyz': True,
  'interaction_xyz': [-2238.681792653615, -40.43783288906502, 1664.8890219981176],
  'parent': 'NuMu',
  'parent_creation_energy': 8326.961594015756,
  'parent_pdg': 14,
  'reaction': 'NuMu -> MuMinus + Hadrons'},
 {'children': ['PairProd', 'MuMinus'],
  'children_creation_energy': [2.666167089665117, 4413.79454352998],
  'children_length': [1805.3992162899892, 0.6925663850363344],
  'children_pdg': [-2000001003, 13],
  'children_share_prop_xyz': True,
  'parent': 'MuMinus',
  'parent_creation_energy': 7025.321571476298,
  'parent_creation_point_xyz': [-2238.681792653615, -40.43783288906502, 1664.8890219981176],
  'parent_length': 4532.132120056887,
  'parent_pdg': 13,
  'prop_xyz': [-983.8875645392309, 170.56547712610853, 384.09420080917437],
  'reaction

In [6]:
print(selected_frame["I3MCTree_postprop"])

[I3MCTree:
  13 NuMu (-2238.68m, -40.4378m, 1664.89m) (44.7929deg, 189.399deg) 0ns 8326.96GeV 0m Primary
    7 MuMinus (-2238.68m, -40.4378m, 1664.89m) (44.806deg, 189.543deg) 0ns 7025.32GeV 4532.13m Dark
      17 MuMinus (-985.278m, 170.332m, 385.514m) (44.8159deg, 189.537deg) 6015.49ns 4416.98GeV 2.00095m Null
      18 PairProd (-983.888m, 170.565m, 384.094m) (44.8159deg, 189.537deg) 6022.16ns 2.66617GeV 1805.4m Null
      19 MuMinus (-983.888m, 170.565m, 384.094m) (44.8157deg, 189.537deg) 6022.16ns 4413.79GeV 0.692566m Null
      20 PairProd (-983.406m, 170.646m, 383.603m) (44.8157deg, 189.537deg) 6024.47ns 1.88859GeV 1806.09m Null
      21 MuMinus (-983.406m, 170.646m, 383.603m) (44.8142deg, 189.538deg) 6024.47ns 4411.73GeV 5.03287m Null
      22 DeltaE (-979.908m, 171.234m, 380.033m) (44.8142deg, 189.538deg) 6041.26ns 0.728863GeV 1811.12m Null
      23 MuMinus (-979.908m, 171.234m, 380.033m) (44.8142deg, 189.538deg) 6041.26ns 4409.7GeV 0.0631028m Null
      24 DeltaE (-979.864m, 1